In [ ]:
%load_ext autoreload
%autoreload 2

# Design

> What this assay can see: the vector, the library, and the limits they set.

In [ ]:
#| default_exp design

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import re
import numpy as np
import pandas as pd
from randseq.coding import encode_library, _pattern_window_codes, decode_motif_codes
from randseq.utils import revcomp, get_patterns, _custom_motif_to_regex

In [ ]:
#| hide
import os
from randseq.core import find_restricted_motifs
from randseq.utils import calculate_log2fc
from randseq.example_data import get_example_data_dir, get_example_backbone

In [ ]:
#| hide
data_path = get_example_data_dir()

## Can a given motif be detected at all?

A motif is called by contrasting the plasmids that carry it against the plasmids that do not, so
**both groups have to exist and both have to be large enough**. Whether they do is settled before
any data is collected, by the vector and the library.

An enzyme cuts the molecule, not the insert. So write `c_b` for the copies of the site in the
**backbone** and `c_i` for the copies in the **insert**: what a plasmid presents is `c_b + c_i`,
and the question comes in two stages.

### Stage 1 — is the site already in the vector?

If `c_b > 0`, every member of the library carries at least one copy and there is no motif-free
group at all. What that implies is not one thing:

| the system | what a backbone copy does |
|---|---|
| **one site is enough to restrict** — the common case for Type I and Type II RM | the backbone already saturates it. `c_i` adds nothing the assay can see and the motif is **effectively invisible** |
| **interference scales with motif density** — as it does for BREX | the contrast comes from `c_i` varying *on top of* `c_b`. Detection still works, and can even **depend** on the backbone copies to bring the plasmid near the threshold |
| **something else limits activity** | removing the backbone copies need not restore a signal at all |

So a backbone copy is a reason to **distrust a negative**, not a proof of one. Nothing below ever
calls a motif undetectable; it reports copy numbers and leaves the reading to you.

### Stage 2 — if the vector is clean, is the split big enough?

With `c_b = 0` the contrast is between `c_i = 0` and `c_i >= 1`, and that is set by the insert
alone. The probability that a random insert of length `L` contains a motif with `k` defined
bases, on either strand, is `1 - (1 - 4**-k)**(2*(L-k+1))`:

In [ ]:
def carrier_fraction(k, L):
    """Chance that a random insert of length `L` carries a given motif with `k` defined
    bases, on either strand."""
    return 1 - (1 - 4.0 ** -k) ** (2 * (L - k + 1))

def library_shape(counts):
    "Library size and typical insert length, read off a count table."
    return len(counts), int(pd.Series([len(s) for s in counts.index]).median())

ks = range(3, 14)
tables = {}
for f in ["countsTable.csv.gz", "counts_panel.csv.gz"]:
    N, L = library_shape(pd.read_csv(os.path.join(data_path, f), index_col=0))
    p = np.array([carrier_fraction(k, L) for k in ks])
    tables[f"{N:,} inserts of {L} nt"] = pd.DataFrame(
        {"carriers": np.round(p * N).astype(int),
         "non-carriers": np.round((1 - p) * N).astype(int)},
        index=pd.Index(ks, name="defined bases"))

pd.concat(tables, axis=1)      # the two libraries that ship with the package

11,629 inserts of 20 nt  ... 34,217 inserts of 150 nt
                             carriers  ...             non-carriers
defined bases                          ...                         
3                                5032  ...                      323
4                                1449  ...                    10827
5                                 358  ...                    25724
6                                  85  ...                    31878
7                                  20  ...                    33621
8                                   5  ...                    34068
9                                   1  ...                    34180
10                                  0  ...                    34208
11                                  0  ...                    34215
12                                  0  ...                    34216
13                                  0  ...                    34217

[11 rows x 4 columns]

Read each pair of columns together: **both** have to stay large.

- **Too few defined bases and almost every plasmid carries the motif.** No motif-free baseline is
  left, so even a site that destroys every carrier comes back at the background rate.
- **Too many and almost none does.** A motif needs `flexible_motif_min_support = 5` carriers to be
  scored at all, and few carriers cannot reach a small q-value (below).

Where that usable window falls is a property of the library, not of the method. The 150 nt
library runs from about `k = 5` to about `k = 9`, which covers every pattern `get_patterns()`
searches: contiguous sites of 5–7 bases and gapped sites (the bipartite Type I sites among them)
of 5–8 defined bases. The 20 nt library, a third the size with inserts a seventh as long, is down
to 20 carriers at `k = 7` and 5 at `k = 8`. It can still call a 6 bp Type II site; an 8-base
bipartite site lying inside the insert is beyond it, so a negative for one means nothing.
Lengthening the insert raises the top of the window and erodes the bottom; enlarging the library
raises the top alone.

These counts are for sites wholly inside the insert. The scan also reads the flanks, so a site
that a flank half-completes has many more carriers. JJ1886's `ATACNNNNGTG` finds `ATAC` and its
spacer in the left flank and needs only `GTG` from the insert: 192 carriers on the 20 nt library,
against the 20 expected for 7 random bases.

Support is not the only floor. A motif is called when its binomial p-value against its carriers'
background — the depletion rate of plasmids of the same GC content in this sample — clears the
Benjamini-Hochberg cutoff over every motif the scan tested — between 10⁻⁶ and 2 × 10⁻⁴ on the
strains that ship with the package, smaller the fewer motifs deplete — and when the carriers show,
with 95% confidence, that at least half of them are depleted (`fraction_depleted_lower`). The fewest
carriers that clear both, at a cutoff of 10⁻⁴, by how depleted the motif is and how noisy the
sample:

In [ ]:
from scipy.stats import beta, binom

In [ ]:

def min_carriers(fraction_depleted, background, cutoff=1e-4, score_thr=0.5):
    "Fewest carriers, from min_support = 5, at which a motif this depleted is called."
    n = np.arange(5, 2001)
    k = np.ceil(fraction_depleted * n)
    ok = (binom.sf(k - 1, n, background) <= cutoff) & (beta.ppf(0.05, k, n - k + 1) >= score_thr)
    return int(n[ok][0]) if ok.any() else None

fracs, backgrounds = (1.0, 0.8, 0.6), (0.05, 0.1, 0.3)
pd.DataFrame([[min_carriers(f, b) for b in backgrounds] for f in fracs],
             index=pd.Index(fracs, name="fraction depleted"),
             columns=pd.Index(backgrounds, name="background rate"))

background rate,0.05,0.10,0.30
fraction depleted,,,
1.0,5,5,8
0.8,8,8,13
0.6,62,62,62


Most shipped strains deplete 0.09–0.19 of their plasmids at the default threshold, and a motif's
background is close to that unless its carriers' GC content is unusual; 381A, the noisy one,
depletes 0.30. For a strong site `min_support` is the binding floor: 5 carriers suffice when all are
destroyed, 8 when 80% are, 8–13 depending on the background. A weak site where 60% of carriers are
depleted needs 62 whatever the background: below that its carriers cannot show that most of them
are depleted, which a 7-base site, with 20 carriers on the 20 nt library, never can. "No motif
found" is a strong negative for strong systems and a weak one for weak systems on a small
library.

Run `carrier_fraction` on your own `N` and `L` before reading anything into a motif you did not
find.

Stage 1 bites hardest exactly where stage 2 is most comfortable. A short site is easy to detect
from the insert and almost certain to be in the vector already; a long bipartite site is the
reverse. The next section measures stage 1.

## What the backbone hides

Stage 1 is a property of the vector, so it can be answered exactly once the vector sequence is
known — no data required.

`pMin` is a 2.2 kb mobilisable vector. The library is cloned in place of a `ccdB` cassette, so
what every library member shares is the 2,214 bp running from one side of the insert round to the
other, which is what ships as `pMin_backbone.fasta`.

In [ ]:
backbone = get_example_backbone()
print(f"{len(backbone)} bp; starts at the right flank, ends at the left flank")
print(f"  {backbone[:20]} ... {backbone[-20:]}")

2214 bp; starts at the right flank, ends at the left flank
  ACTAGAGCACCAGAAGTCTA ... GTCTAGGGCGGCGGTAAAAC


In [ ]:
#| export
def _motif_site_count(sequence, motif, circular=False):
    """How many times `motif` occurs as a site in `sequence`, on either strand.

    A site is counted once however it reads, which is what an enzyme sees, so a palindromic
    motif is not counted twice. `circular=True` also finds sites spanning the origin.
    """
    seq = sequence.upper()
    if circular and len(seq) >= len(motif):
        seq = seq + seq[:len(motif) - 1]
    pat = _custom_motif_to_regex(motif)
    n = len(re.findall(f"(?={pat})", seq))
    rc_motif = revcomp(motif)
    if rc_motif != motif:                       # not a palindrome: the other reading is distinct
        n += len(re.findall(f"(?={_custom_motif_to_regex(rc_motif)})", seq))
    return n

In [ ]:
# A site is counted on either strand, once: an enzyme sees one site however it reads.
test_eq(_motif_site_count("AAAGGTCTCAAA", "GGTCTC"), 1)
test_eq(_motif_site_count("AAAGAGACCAAA", "GGTCTC"), 1)        # the same site, other strand
test_eq(_motif_site_count("GGTCTCAAGAGACC", "GGTCTC"), 2)
test_eq(_motif_site_count("AAAAAAAA", "GGTCTC"), 0)
# A palindrome reads identically both ways and is one site, not two.
test_eq(_motif_site_count("AAAGGTACCAAA", "GGTACC"), 1)
# IUPAC codes and N are honoured, since called motifs use them.
test_eq(_motif_site_count("AAACACTTTTGTATAAA", "CACNNNNGTAT"), 1)
test_eq(_motif_site_count("AAACACTTTTGTACAAA", "CACNNNNGTAY"), 1)
# A site spanning the origin is found only when asked: a library backbone is linear.
test_eq(_motif_site_count("CTCAAAAAAAAAGGT", "GGTCTC", circular=False), 0)
test_eq(_motif_site_count("CTCAAAAAAAAAGGT", "GGTCTC", circular=True), 1)

In [ ]:
#| export
def backbone_motif_counts(backbone, patterns, circular=False):
    """Every motif of `patterns` that the backbone already carries, and how many copies.

    `circular=False` is right for a library backbone, which is a linear stretch running from one
    side of the insert round to the other -- the insert closes it, so it is not circular on its
    own. Pass `circular=True` for a whole plasmid.

    Returns 'motif', 'pattern' and 'copies', one row per motif carried at least once.
    """
    rows = []
    for pattern in patterns:
        d1, spacer, d2 = pattern
        width = d1 + spacer + d2
        seq = backbone.upper()
        if circular and len(seq) >= width:
            seq = seq + seq[:width - 1]
        if len(seq) < width:
            continue
        f, r = encode_library([seq])
        n_codes = 4 ** (d1 + d2)
        counts = np.zeros(n_codes + 1, dtype=np.int64)
        for arr in (f, r):
            codes = _pattern_window_codes(arr, pattern).ravel()
            counts += np.bincount(codes, minlength=n_codes + 1)
        present = np.flatnonzero(counts[:n_codes])
        if not len(present):
            continue
        for motif, c in zip(decode_motif_codes(present, pattern), counts[present]):
            # a palindrome reads identically on both strands, so each site was counted twice
            rows.append(dict(motif=motif, pattern=str(pattern),
                             copies=int(c) // 2 if revcomp(motif) == motif else int(c)))
    return pd.DataFrame(rows, columns=["motif", "pattern", "copies"])

In [ ]:
# On pMin: HS's BREX site GGTAAG is in the backbone twice, listed under both spellings.
_bb6 = backbone_motif_counts(backbone, [(6, 0, 0)]).set_index("motif")
test_eq(_bb6.loc[["GGTAAG", "CTTACC"], "copies"].tolist(), [2, 2])
_bb6.sort_values("copies", ascending=False).head()

,pattern,copies
motif,,
TTTTTT,"(6, 0, 0)",13
AAAAAA,"(6, 0, 0)",13
TTTTTC,"(6, 0, 0)",10
GAAAAA,"(6, 0, 0)",10
AAAAAG,"(6, 0, 0)",9


In [ ]:
#| hide
# The scan-based counter and the regex-based one must agree, motif for motif.
_bb = "".join(np.random.default_rng(0).choice(list("ACGT"), 400))
_c = backbone_motif_counts(_bb, [(6, 0, 0), (3, 4, 4)])
assert len(_c) and (_c.copies > 0).all()
for _r in _c.sample(25, random_state=0).itertuples():
    test_eq(_r.copies, _motif_site_count(_bb, _r.motif))
# A backbone with no room for the pattern yields nothing rather than raising.
test_eq(len(backbone_motif_counts("ACGT", [(6, 0, 0)])), 0)

In [ ]:
#| export
def backbone_masking_summary(backbone, patterns, circular=False):
    """Per pattern, how much of the searchable motif space the backbone already occupies.

    A motif the backbone carries is one the assay is unlikely to report for this vector, so this
    is the coverage of the search space -- the share of possible answers ruled out before any
    data is collected.
    """
    counts = backbone_motif_counts(backbone, patterns, circular=circular)
    carried = counts.groupby("pattern").size() if len(counts) else pd.Series(dtype=int)
    rows = []
    for pattern in patterns:
        space = 4 ** (pattern[0] + pattern[2])
        n = int(carried.get(str(pattern), 0))
        rows.append(dict(pattern=str(pattern), defined_bases=pattern[0] + pattern[2],
                         motifs=space, carried=n, fraction=n / space))
    return pd.DataFrame(rows)

In [ ]:
# The summary counts, per pattern, the motifs backbone_motif_counts lists, out of 4**k possible.
_s = backbone_masking_summary(_bb, [(6, 0, 0)])
test_eq(int(_s.motifs.iloc[0]), 4096)
test_eq(int(_s.carried.iloc[0]), len(_c[_c.pattern == "(6, 0, 0)"]))
test_eq(float(_s.fraction.iloc[0]), _s.carried.iloc[0] / 4096)

On the architectures `get_patterns()` searches:

In [ ]:
summary = backbone_masking_summary(backbone, get_patterns())
by_size = (summary.groupby("defined_bases")
                  .agg(patterns=("pattern", "size"), mean_masked=("fraction", "mean")))
print(by_size.to_string(float_format=lambda x: f"{x:.1%}"))
print(f"\nover the whole searched space: {summary.carried.sum():,} of {summary.motifs.sum():,} "
      f"motifs = {summary.carried.sum() / summary.motifs.sum():.1%} already carried")

               patterns  mean_masked
defined_bases                       
5                     9        97.6%
6                    17        62.9%
7                     9        22.8%
8                     8         6.4%

over the whole searched space: 119,913 of 750,592 motifs = 16.0% already carried


**Masking is almost entirely a short-motif problem.** With this vector 98% of 5-base sites are
already present and nearly two in three 6-base sites, against 23% of 7-base and 6% of 8-base
ones. So with pMin the method can recover long bipartite Type I specificities far more readily
than short Type II ones, whatever the strain.

It also says what a second vector buys: a motif is only truly out of reach if **every** backbone
in use carries it, and two unrelated 2.2 kb vectors share far fewer sites than either carries
alone.

### Reading a result against the backbone

`annotate_backbone` adds the copy number to a result table. A reported motif normally comes back
with zero copies, since a motif the vector carries is usually not detectable in the first place.
A non-zero count is not a mistake: it is the density-dependent case, and a signal to look closer.

In [ ]:
#| export
def annotate_backbone(results_df, backbone, circular=False):
    """Add `backbone_copies` to a result table: copies of each reported motif in the backbone.

    0 is the expected value for anything the pipeline reported, since a motif the backbone
    carries is usually not detectable in the first place. A non-zero count is worth looking at:
    it means the motif was called *despite* the vector already carrying it, which happens when
    interference is density-dependent.
    """
    if results_df is None or len(results_df) == 0 or "motif" not in results_df.columns:
        return results_df
    out = results_df.copy()
    out["backbone_copies"] = [_motif_site_count(backbone, m, circular) for m in out["motif"]]
    return out

In [ ]:
# JJ1886's three calls on the panel, and the BREX site of E. coli HS
_called = pd.DataFrame({"motif": ["RTACNNNNGTG", "AAAGNNNNGTT", "GGTCTC", "GGTAAG"]})
_annotated = annotate_backbone(_called, backbone)
_annotated

,motif,backbone_copies
0,RTACNNNNGTG,0
1,AAAGNNNNGTT,0
2,GGTCTC,0
3,GGTAAG,2


In [ ]:
# JJ1886's sites are absent from the vector; the BREX site is there twice.
test_eq(_annotated.backbone_copies.tolist(), [0, 0, 0, 2])
# The input table is not modified, and a table with nothing to annotate comes back as it was.
assert "backbone_copies" not in _called.columns
test_eq(len(annotate_backbone(_called.iloc[:0], backbone)), 0)
assert annotate_backbone(None, backbone) is None

JJ1886's sites are absent from the vector, which is part of why they could be called. `GGTAAG`, the
BREX site of *E. coli* HS, is there twice: the case where a backbone motif is detected anyway,
because interference scales with density (below).

## Failure modes, on real strains

The shipped panel exists so that each of these can be run, not just read. See
`example_data/counts_panel_key.csv`.

**Weak systems need a permissive threshold.** BREX interferes more weakly than a
restriction–modification system: *E. coli* HS has a known BREX system recognising `GGTAAG`, and
at `log2FC < -1` the pipeline calls **nothing** for it. That is why the default is `-0.5`, applied
after each sample is centred on its median: without the centring, a threshold that permissive
floods strains whose log2FC is shifted as a whole (see `00_core`). At `-0.5` HS calls `GGTAAG`.
A negative result at a stricter threshold says nothing about a weakly interfering system.

In [ ]:
counts_panel = pd.read_csv(os.path.join(data_path, "counts_panel.csv.gz"), index_col=0)
log2fc_panel = calculate_log2fc(counts_panel, reference_column="Control_R1", count_threshold=20)
left_panel, right_panel = "GTCTAGGGCGGCGGTAAAAC", "ACTAGAGCACCAGAAGTCTA"

def called_motifs(strain, **kwargs):
    "Motifs `find_restricted_motifs` calls (q < 0.05) for one strain of the panel."
    return list(find_restricted_motifs(log2fc_panel[strain], left_panel, right_panel, **kwargs).motif)

# -1 is too strict to see HS's BREX site; the default -0.5 is not.
test_eq(called_motifs("HS_R1", flexible_motif_log2fc_thr=-1), [])
test_eq(called_motifs("HS_R1"), ["GGTAAG"])

1 candidate(s) across all patterns; 1 remain after step 3.
1 motif(s) scored:
  CGGCNGCCG (pattern (4, 1, 4)), FracDep: 0.50, AvgFC: -0.30, N: 6
0 motif(s) called.
11 candidate(s) across all patterns; 3 remain after step 3.
2 candidate(s) whose score was borrowed:
    AGGTCTC <- CTTACC (without them: n=10, score=0.40)
    CGGCNGCCG <- CTTACC (without them: n=5, score=0.40)
1 motif(s) scored:
  GGTAAG (pattern (6, 0, 0)), FracDep: 0.64, AvgFC: -0.65, N: 1879
1 motif(s) called.


Scanning patterns: 100%|##########| 43/43 [00:03<00:00, 11.07it/s]


**Specificity, demonstrated isogenically.** EDL933 calls `CACNNNNNNNCTGG`; its isogenic RM
knockout `EDL933_dRM` calls nothing. A negative result from an unrelated strain proves much less
than a matched pair.

In [ ]:
test_eq(called_motifs("EDL933_R1"), ["CACNNNNNNNCTGG"])
test_eq(called_motifs("EDL933_dRM_R1"), [])

9 candidate(s) across all patterns; 1 remain after step 3.
1 motif(s) scored:
  CACNNNNNNNCTGG (pattern (3, 7, 4)), FracDep: 0.91, AvgFC: -1.16, N: 277
1 motif(s) called.
0 motif(s) scored:
0 motif(s) called.


Scanning patterns: 100%|##########| 43/43 [00:02<00:00, 14.85it/s]


**A motif already in your vector usually cannot be detected.** That is stage 1 above: every
plasmid carries it, so there is no motif-free baseline. The screen in this work hit exactly this
and had to change backbone to recover a masked Type I motif, so check your backbone before
believing a negative.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| hide